<font>
<div dir=ltr align=center>
<img src='https://cdn.freebiesupply.com/logos/large/2x/sharif-logo-png-transparent.png' width=150 height=150> <br>
<font color=0F5298 size=6>
Natural Language Processing<br>
<font color=2565AE size=4>
Computer Engineering Department<br>
Spring 2025<br>
<font color=3C99D size=4>
Workshop 4 - LLMs<br>
<font color=696880 size=3>
<a href='https://language.ml'>https://language.ml</a><br>
info [AT] language [dot] ml

# 1. Introduction

In this notebook, we will explore Hugging Face, a popular way to interact with large language models. We’ll see how to install, configure, and run basic examples in the framework.

# 2. [Hugging Face](https://github.com/huggingface)

> **Tip:** Every error you encounter while training or fine-tuning is a chance to contribute back-consider turning that fix into a pull request!

## Login

For login, first create a token at [Hugging Face](https://huggingface.co/settings/tokens). You can adjuust the token's permissions based on your needs, but for most tasks, the default settings are sufficient.

Some models/datasets may require you to accept their license terms before you can use them. You can do this by visiting the model's page on Hugging Face like [mozilla-foundation/common_voice_17_0](https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0). Or, some models/datasets need approval from the model owner, like [meta-llama/Llama-4-Scout-17B-16E-Instruct](https://huggingface.co/meta-llama/Llama-4-Scout-17B-16E-Instruct). In this case, you can request access.

You can use the `notebook_login()` function to log in interactively, or you can set the `HF_TOKEN` environment variable with your token string. After logging in, your token will be store in somewhere like `~/.cache/huggingface/token`.

In [50]:
from huggingface_hub import notebook_login

# notebook_login()

## Loading a Dataset

We'll use the [GLUE](https://huggingface.co/datasets/nyu-mll/glue) dataset as an example. The GLUE dataset is a collection of various NLP tasks, and it is widely used for benchmarking models.

The dataset stores in `~/.cache/huggingface/datasets/nyu-mll___glue` directory. When you do some operations (e.g. map or cast) on the dataset, it will be cached in the directory.

In [51]:
from datasets import load_dataset, DatasetDict

dataset = load_dataset('nyu-mll/glue', name='cola')
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})

In [3]:
type(dataset)

datasets.dataset_dict.DatasetDict

In [4]:
dataset = load_dataset('nyu-mll/glue', name='cola', split='train+test')
dataset

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 9614
})

In [5]:
type(dataset)

datasets.arrow_dataset.Dataset

In [6]:
# You can also PR the following IDE warning
dataset = load_dataset('nyu-mll/glue', name='cola', split=['train', 'test'])
dataset

[Dataset({
     features: ['sentence', 'label', 'idx'],
     num_rows: 8551
 }),
 Dataset({
     features: ['sentence', 'label', 'idx'],
     num_rows: 1063
 })]

In [7]:
dataset = load_dataset('nyu-mll/glue', name='cola', split={'train': 'train', 'test': 'test'})
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})

In [8]:
dataset = DatasetDict()

dataset['train'] = load_dataset('nyu-mll/glue', name='cola', split='train')
dataset['test'] = load_dataset('nyu-mll/glue', name='cola', split='test')
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})

In [9]:
print('Shape of the dataset:', dataset.shape)

Shape of the dataset: {'train': (8551, 3), 'test': (1063, 3)}


In [10]:
print('Column names:', dataset.column_names)

Column names: {'train': ['sentence', 'label', 'idx'], 'test': ['sentence', 'label', 'idx']}


In [52]:
print('Features:', dataset['train'].features)

Features: {'sentence': Value(dtype='string', id=None), 'label': ClassLabel(names=['unacceptable', 'acceptable'], id=None), 'idx': Value(dtype='int32', id=None)}


In [53]:
dataset['train'][0]

{'sentence': "Our friends won't buy this analysis, let alone the next one we propose.",
 'label': 1,
 'idx': 0}

You can apply some operations on the dataset, like filtering, mapping, and casting. For example, you can filter the dataset to keep only the examples with a specific label.

In [54]:
def filter_func(example):
    return example['label'] == 1

In [56]:
# See args of `filter` function by clicking on the function name in an IDE
# `filter` function `for` across all splits and all examples in each split
filtered_dataset = dataset.filter(filter_func, desc='Filtering positive examples')
filtered_dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 6023
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 721
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 0
    })
})

In [15]:
filtered_dataset = dataset.filter(lambda example: example['label'] == 1, desc='Filtering positive examples')
filtered_dataset

Filtering positive examples:   0%|          | 0/8551 [00:00<?, ? examples/s]

Filtering positive examples:   0%|          | 0/1063 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 6023
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 0
    })
})

In [ ]:
filtered_dataset = dataset.filter(lambda label: label == 1, input_columns=['label'])
filtered_dataset

# Loading a Pre-trained Model & Tokenizer

First, we'll use [cis-lmu/glot500-base](https://huggingface.co/cis-lmu/glot500-base) as an example to get familiar with some concepts and APIs of Hugging Face. This model is a multilingual BERT-based model trained on the [GLot500](https://huggingface.co/datasets/cis-lmu/glot500) dataset, which contains 500 languages including Persian. Check info of the model in the [model card](https://huggingface.co/cis-lmu/glot500-base).

Persian Splits:
- `fas_Arab`: Persian (the generic (macrolanguage) code with Arabic script)
- `pes_Arab`: Persian (the Iranian/Farsi code with Arabic script)

In [57]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('cis-lmu/glot500-base')
tokenizer

XLMRobertaTokenizerFast(name_or_path='cis-lmu/glot500-base', vocab_size=401145, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	401144: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
}
)

In [58]:
from transformers import AutoModelForMaskedLM

model = AutoModelForMaskedLM.from_pretrained('cis-lmu/glot500-base')
model

XLMRobertaForMaskedLM(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(401145, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True

In [19]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained('cis-lmu/glot500-base')
model

If you want to use `XLMRobertaLMHeadModel` as a standalone, add `is_decoder=True.`


XLMRobertaForCausalLM(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(401145, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True

What does this warning mean?
- If you want to use `XLMRobertaLMHeadModel` as a standalone, add `is_decoder=True.`

Answer:
- `XLMRobertaLMHeadModel` is an encoder-only model, which means it is not designed to generate text like decoder models (e.g., GPT-2).
- If you want to use it for text generation, you need to set `is_decoder=True` in the configuration. This will allow the model to generate text by predicting the next token based on the input sequence.

Questions:
- What are encoder-only models and decoder-only models?
- What is masked language modeling (MLM)? Is it self-supervised learning?
- What is causal language modeling (CLM)? Is it self-supervised learning?

In [20]:
from transformers import AutoModel

model = AutoModel.from_pretrained('cis-lmu/glot500-base')
model

Some weights of XLMRobertaModel were not initialized from the model checkpoint at cis-lmu/glot500-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(401145, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine

For famous models there are their own classes (e.g. Config, Model and Tokenizer classes). You can find the documentation in the [Hugging Face documentation](https://huggingface.co/docs) more specifically, [Transformers documentation](https://huggingface.co/docs/transformers/index). You can also use IDE's auto-completion feature.

In [60]:
from transformers import XLMRobertaModel

model = XLMRobertaModel.from_pretrained('cis-lmu/glot500-base')
model

Some weights of XLMRobertaModel were not initialized from the model checkpoint at cis-lmu/glot500-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(401145, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine

In [61]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained('cis-lmu/glot500-base')
model = AutoModelForMaskedLM.from_pretrained('cis-lmu/glot500-base')

text = 'Hello I\'m a <mask> model.'
encoded_input = tokenizer(text, return_tensors='pt')

output = model(**encoded_input)
output

MaskedLMOutput(loss=None, logits=tensor([[[ 2.6283, -0.8172, -1.0258,  ..., -2.1955, -1.0647, -0.8886],
         [ 3.3795,  0.3517,  0.4499,  ..., -3.8742, -1.8115, -1.6796],
         [ 2.4203, -1.5251, -2.2870,  ..., -7.0230, -3.3443, -4.2717],
         ...,
         [-1.6454, -1.0379, -4.6366,  ..., -1.9015, -0.8423, -1.4584],
         [ 6.2019, -0.4570,  4.4269,  ..., -3.9569, -0.6303, -2.8477],
         [ 6.2563, -0.9551, -0.9831,  ..., -3.2835, -1.5957, -1.7238]]],
       grad_fn=<ViewBackward0>), hidden_states=None, attentions=None)

In [62]:
encoded_input

{'input_ids': tensor([[     0,  35378,     87,     25,     39,     10, 401144,   3299,      5,
              2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [24]:
tokenizer.convert_ids_to_tokens(encoded_input['input_ids'][0])

['<s>', '▁Hello', '▁I', "'", 'm', '▁a', '<mask>', '▁model', '.', '</s>']

In [25]:
logits = output.logits
logits.shape

torch.Size([1, 10, 401145])

In [26]:
import torch

# 6 is index of the <mask> token
next_token_logits = output.logits[:, 6, :]
next_token_logits.shape

torch.Size([1, 401145])

In [27]:
next_token_id = torch.argmax(next_token_logits, dim=-1)
next_token_id

tensor([54543])

In [28]:
next_token = tokenizer.decode(next_token_id)
print('Next token:', next_token)

Next token: fashion


In [29]:
from transformers import pipeline

pipe = pipeline('fill-mask', model='cis-lmu/glot500-base')
pipe('Hello I\'m a <mask> model.')

Device set to use mps:0


[{'score': 0.27405598759651184,
  'token': 54543,
  'token_str': 'fashion',
  'sequence': "Hello I'm a fashion model."},
 {'score': 0.055254045873880386,
  'token': 8063,
  'token_str': 'business',
  'sequence': "Hello I'm a business model."},
 {'score': 0.03785160183906555,
  'token': 17473,
  'token_str': 'sexy',
  'sequence': "Hello I'm a sexy model."},
 {'score': 0.03608034923672676,
  'token': 34923,
  'token_str': 'beautiful',
  'sequence': "Hello I'm a beautiful model."},
 {'score': 0.03388901799917221,
  'token': 62607,
  'token_str': 'beauty',
  'sequence': "Hello I'm a beauty model."}]

# Training

In [30]:
dataset = load_dataset('nyu-mll/glue', name='cola')
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})

In [31]:
from transformers import AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained('cis-lmu/glot500-base')
model = AutoModelForSequenceClassification.from_pretrained('cis-lmu/glot500-base', num_labels=2)
model

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at cis-lmu/glot500-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(401145, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

In [32]:
def tokenize_function(examples):
    return tokenizer(
        examples['sentence'],
        truncation=True,
        padding='max_length',  # or 'longest' + DataCollator for dynamic
        max_length=128
    )

In [33]:
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['sentence', 'idx']  # keep only input_ids, attention_mask, label
)
tokenized_datasets

Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1063
    })
})

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./cola_trainer',      # where checkpoints & logs go
    num_train_epochs=3,               # number of epochs
    per_device_train_batch_size=16,   # batch size per GPU/CPU
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',            # run evaluation at end of each epoch
    save_strategy='epoch',            # checkpoint at end of each epoch
    logging_dir='./logs',             # tensorboard logs (if you use tb)
    logging_steps=100,
    load_best_model_at_end=True,      # (optional) keep best checkpoint by eval metric
    metric_for_best_model='accuracy', # which metric to monitor
    greater_is_better=True            # if higher metric is better
)

- If you use `padding='max_length'` in the tokenizer, a simple data collator isn’t required.
- But if you want dynamic padding to the longest sequence in each batch, use `DataCollatorWithPadding`.

In [37]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [41]:
import numpy as np
from sklearn.metrics import accuracy_score, matthews_corrcoef

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # For classification, logits shape = (batch_size, num_labels)
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    # Matthews Correlation Coefficient is a common metric for CoLA.
    # We can set metric_for_best_model='matthews_corrcoef' in TrainingArguments if you want to load the best model by MCC.
    mcc = matthews_corrcoef(labels, predictions)

    return {'accuracy': acc, 'matthews_corrcoef': mcc}

In [44]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'].select(range(1000)),
    eval_dataset=tokenized_datasets['validation'].select(range(1000)),
    tokenizer=tokenizer,                   # needed if you’re logging predictions
    data_collator=data_collator,           # or mlm_collator if MLM
    compute_metrics=compute_metrics        # optional, but recommended for classification
)

/var/folders/tk/kx7c6x1j16gdg23gt5g8mr440000gn/T/ipykernel_8808/1966386508.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [45]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Matthews Corrcoef
1,No log,0.623540,0.691000,0.000000
2,0.667700,0.627759,0.691000,0.000000
3,0.667700,0.623468,0.691000,0.000000


/opt/homebrew/anaconda3/envs/Jupyter/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/homebrew/anaconda3/envs/Jupyter/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [46]:
train_result

TrainOutput(global_step=189, training_loss=0.6543788405322524, metrics={'train_runtime': 60.721, 'train_samples_per_second': 49.406, 'train_steps_per_second': 3.113, 'total_flos': 197333291520000.0, 'train_loss': 0.6543788405322524, 'epoch': 3.0})

In [47]:
metrics = trainer.evaluate(eval_dataset=tokenized_datasets['test'])

metrics

/opt/homebrew/anaconda3/envs/Jupyter/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.0,
 'eval_accuracy': 0.0,
 'eval_matthews_corrcoef': 0.0,
 'eval_runtime': 4.4417,
 'eval_samples_per_second': 239.32,
 'eval_steps_per_second': 7.655,
 'epoch': 3.0}

In [49]:
trainer.save_model('./cola_best_model')
tokenizer.save_pretrained('./cola_best_model')

('./cola_best_model/tokenizer_config.json',
 './cola_best_model/special_tokens_map.json',
 './cola_best_model/sentencepiece.bpe.model',
 './cola_best_model/added_tokens.json',
 './cola_best_model/tokenizer.json')

In [ ]:
TrainingArguments(
    gradient_accumulation_steps=4,
    callbacks=[]
)